In [67]:
!pip install requests
!pip install feedparser
!pip install feedparser
!pip install feedparser beautifulsoup4 requests
!pip install pandas requests feedparser beautifulsoup4 trafilatura lxml

In [68]:
import logging
from urllib.parse import quote

import feedparser
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup


# Silence extraction warnings
logging.getLogger("trafilatura").setLevel(logging.ERROR)
logging.getLogger("trafilatura.core").setLevel(logging.ERROR)
logging.getLogger("trafilatura.utils").setLevel(logging.ERROR)


# Terminal colors
RESET = "\033[0m"
GREEN = "\033[92m"
RED = "\033[91m"
BLUE = "\033[94m"
CYAN = "\033[96m"
BOLD = "\033[1m"


PEERS = {
    "AAPL": ["MSFT", "GOOGL", "AMZN", "META", "NVDA", "QQQ"],
    "MSFT": ["AAPL", "GOOGL", "AMZN", "META", "NVDA", "QQQ"],
    "NVDA": ["AMD", "TSM", "AVGO", "QCOM", "SOXX", "SMH"],
    "AMD": ["NVDA", "INTC", "QCOM", "AVGO", "SOXX", "SMH"],
    "TSLA": ["RIVN", "LCID", "NIO", "GM", "F", "XLY"],
    "META": ["GOOGL", "SNAP", "PINS", "RDDT", "QQQ", "XLC"],
    "AMZN": ["WMT", "COST", "SHOP", "EBAY", "XLY", "QQQ"],
    "GOOGL": ["META", "MSFT", "AAPL", "QQQ", "XLC", "SNAP"],
    "QQQ": ["SPY", "IWM", "SMH", "SOXX", "AAPL", "MSFT"],
    "SPY": ["QQQ", "IWM", "DIA", "VOO", "IVV"],
    "IWM": ["SPY", "QQQ", "DIA"],
    "SOXL": ["SOXX", "SMH", "NVDA", "AMD", "AVGO", "TSM"],
    "SOXX": ["SMH", "NVDA", "AMD", "AVGO", "TSM", "QCOM"],
    "SMH": ["SOXX", "NVDA", "AMD", "AVGO", "TSM", "QCOM"],
    "AVGO": ["NVDA", "AMD", "QCOM", "TSM", "SOXX", "SMH"],
    "TSM": ["NVDA", "AMD", "AVGO", "QCOM", "SOXX", "SMH"],
    "QCOM": ["AVGO", "AMD", "NVDA", "TSM", "SOXX", "SMH"],
    "RIVN": ["TSLA", "LCID", "NIO", "GM", "F"],
    "LCID": ["TSLA", "RIVN", "NIO"],
    "NIO": ["TSLA", "RIVN", "LCID"],
    "PLTR": ["SNOW", "CRM", "MDB", "AI", "QQQ"],
    "SNOW": ["MDB", "CRM", "PLTR", "DDOG", "QQQ"],
    "CRM": ["NOW", "MDB", "SNOW", "PLTR", "QQQ"],
}
DEFAULT_PEERS = ["SPY", "QQQ", "IWM", "AAPL", "MSFT", "NVDA"]

LEVERAGED = {
    "SOXL", "TQQQ", "SQQQ", "SPXL", "SPXS",
    "LABU", "LABD", "TECL", "FNGU", "FNGD",
    "NVDL", "NVDX", "TSLL", "TSLQ", "BULZ", "BERZ"
}


#Display
def paint(text, color):
    return f"{color}{text}{RESET}"


def good(text):
    return paint(text, GREEN)


def bad(text):
    return paint(text, RED)


def info(text):
    return paint(text, BLUE)


def title(text):
    return paint(text, CYAN + BOLD)


def signed_pct(value):
    return f"{value:+.2f}%"


def icon_for(trend_label, setup_label):
    if trend_label == "Bull market-like condition" or setup_label in {"Support Bounce", "Breakout Forming", "Bullish Pullback"}:
        return "🟢"
    if trend_label == "Bear market-like condition" or setup_label in {"Weak Near Support", "Bearish / Weak Trend"}:
        return "🔴"
    return "🔵"


def peer_icon(setup_label):
    if setup_label in {"Support Bounce", "Breakout Forming", "Bullish Pullback"}:
        return "🟢"
    if setup_label in {"Weak Near Support", "Bearish / Weak Trend"}:
        return "🔴"
    return "🔵"


#News
def score_text_sentiment(text):
    positive = [
        "beat", "beats", "growth", "strong", "surge", "surges", "profit", "profits",
        "record", "upgrade", "upgrades", "gain", "gains", "positive", "expansion",
        "bullish", "outperform", "recovery", "improved", "improves", "partnership",
        "launch", "launched", "success", "successful", "higher", "rise", "rises",
        "raised guidance", "approval", "approves", "contract win", "buyback",
        "acquisition", "demand", "rebound", "momentum"
    ]
    negative = [
        "miss", "misses", "loss", "losses", "weak", "drop", "drops", "decline",
        "declines", "lawsuit", "probe", "investigation", "downgrade", "downgrades",
        "fraud", "recall", "delay", "delays", "cut", "cuts", "negative", "bearish",
        "fall", "falls", "lower", "warning", "risk", "risks", "debt",
        "cuts guidance", "regulatory", "tariff", "tariffs", "selloff", "offering",
        "dilution", "missed estimates", "layoffs"
    ]

    text = str(text).lower()
    pos = sum(text.count(word) for word in positive)
    neg = sum(text.count(word) for word in negative)
    return pos - neg


def extract_story_text(url, rss_summary=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    text = ""

    try:
        raw = trafilatura.fetch_url(url)
        if raw:
            parsed = trafilatura.extract(raw, url=url, include_comments=False, include_tables=False)
            if parsed and len(parsed.strip()) > 400:
                text = parsed.strip()
    except Exception:
        pass

    if len(text) < 400:
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
                tag.extract()

            candidates = soup.select("article, main, [role='main']")
            best = ""

            for block in candidates:
                paragraphs = block.find_all("p")
                combined = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
                if len(combined) > len(best):
                    best = combined

            if len(best) > 300:
                text = best
            else:
                paragraphs = soup.find_all("p")
                text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
        except Exception:
            pass

    if len(text.strip()) < 150 and rss_summary:
        text = rss_summary

    return text.strip()


def background_news_score(symbol):
    query = quote(f"{symbol} stock OR earnings OR guidance OR outlook")
    url = f"https://news.google.com/rss/search?q={query}"
    feed = feedparser.parse(url)

    total = 0
    if not getattr(feed, "entries", None):
        return total

    for entry in feed.entries[:5]:
        headline = entry.get("title", "")
        link = entry.get("link", "")
        summary = entry.get("summary", "")

        try:
            article = extract_story_text(link, summary)
            combined = f"{headline}. {summary}. {article}"
        except Exception:
            combined = f"{headline}. {summary}"

        total += score_text_sentiment(combined)

    return total


#Market Data
def load_history(symbol):
    stooq_symbol = f"{symbol.lower()}.us"
    url = f"https://stooq.com/q/d/l/?s={stooq_symbol}&i=d"
    df = pd.read_csv(url)

    if df.empty or "Close" not in df.columns:
        return None

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Open", "High", "Low", "Close"])
    if df.empty or len(df) < 20:
        return None

    return df


def rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def is_leveraged(symbol):
    return symbol.upper() in LEVERAGED


#Analysis Logic
def classify_setup(close, ema20, ema50, rsi_value, vol_ratio, news_score, breakout20, support20):
    if pd.isna(rsi_value):
        return "Not enough data"

    support_gap = ((close - support20) / support20) * 100 if support20 else 999
    breakout_gap = ((breakout20 - close) / close) * 100 if close else 999

    if support_gap <= 2.5:
        if rsi_value >= 45 and news_score >= 0:
            return "Support Bounce"
        if rsi_value < 45:
            return "Weak Near Support"
        return "Neutral"

    if breakout_gap <= 2.5:
        if close > ema20 and vol_ratio >= 0.9 and news_score >= 0:
            return "Breakout Forming"
        return "Near Breakout"

    if close > ema20 and ema20 > ema50 and 50 <= rsi_value <= 68 and news_score >= 0:
        return "Bullish Pullback"

    if close < ema20 and ema20 < ema50 and 32 <= rsi_value <= 50 and news_score <= 0:
        return "Bearish / Weak Trend"

    return "Neutral"


def build_trade_plan(symbol, close, support20, breakout20, ema20, atr_like, setup):
    atr_like = max(float(atr_like), close * 0.01)

    if is_leveraged(symbol):
        rr_short, rr_long = 0.8, 1.2
        cap_short, cap_long = 0.035, 0.06
    else:
        rr_short, rr_long = 1.0, 1.5
        cap_short, cap_long = 0.04, 0.07

    if setup == "Support Bounce":
        buy = close
        stop = support20 - (0.5 * atr_like)

    elif setup in {"Breakout Forming", "Near Breakout"}:
        buy = breakout20 * 1.002
        stop = breakout20 - (0.75 * atr_like)

    elif setup == "Bullish Pullback":
        buy = min(close, ema20 * 1.01)
        stop = buy - atr_like

    else:
        support_gap = ((close - support20) / support20) * 100 if support20 else 999
        breakout_gap = ((breakout20 - close) / close) * 100 if close else 999

        if support_gap <= 3.0:
            buy = close
            stop = support20 - (0.5 * atr_like)
        elif breakout_gap <= 3.0:
            buy = breakout20 * 1.002
            stop = breakout20 - (0.75 * atr_like)
        else:
            buy = close
            stop = close - atr_like

    if stop >= buy:
        stop = buy - max(atr_like, close * 0.01)

    risk = max(buy - stop, close * 0.01)
    t1_risk = buy + (rr_short * risk)
    t2_risk = buy + (rr_long * risk)

    if setup == "Support Bounce":
        target1 = min(t1_risk, ema20 if ema20 > buy else buy * (1 + cap_short))
    else:
        target1 = min(t1_risk, buy * (1 + cap_short))

    target2 = min(t2_risk, buy * (1 + cap_long))

    return {
        "buy_price": buy,
        "stop_loss": stop,
        "sell_target_1": target1,
        "sell_target_2": target2,
    }


def analyze(symbol, use_news=True):
    df = load_history(symbol)
    if df is None:
        return None

    df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["RSI14"] = rsi(df["Close"], 14)
    df["AVG_VOL20"] = df["Volume"].rolling(20).mean()
    df["RANGE"] = df["High"] - df["Low"]

    row = df.iloc[-1]

    close = row["Close"]
    open_ = row["Open"]
    high = row["High"]
    low = row["Low"]
    volume = row["Volume"]
    date = row["Date"].date()

    day_change = ((close - open_) / open_) * 100 if open_ else 0

    trailing_252 = df.tail(252)
    trailing_40 = df.tail(40)
    trailing_20 = df.tail(20)
    trailing_14 = df.tail(14)

    high_52 = trailing_252["High"].max()
    low_52 = trailing_252["Low"].min()
    recent_low = trailing_40["Low"].min()
    recent_high = trailing_40["High"].max()

    pct_from_low = ((close - recent_low) / recent_low) * 100 if recent_low else 0
    pct_from_high = ((close - recent_high) / recent_high) * 100 if recent_high else 0

    if pct_from_low >= 20:
        trend = "Bull market-like condition"
    elif pct_from_high <= -20:
        trend = "Bear market-like condition"
    else:
        trend = "Neutral / no clear bull-bear condition"

    ema20 = row["EMA20"]
    ema50 = row["EMA50"]
    rsi14 = row["RSI14"]
    avg_vol20 = row["AVG_VOL20"]
    vol_ratio = (volume / avg_vol20) if pd.notna(avg_vol20) and avg_vol20 not in [0, None] else 0

    breakout20 = trailing_20["High"].max()
    support20 = trailing_20["Low"].min()
    atr_like = trailing_14["RANGE"].mean()

    news_score = background_news_score(symbol) if use_news else 0
    setup = classify_setup(close, ema20, ema50, rsi14, vol_ratio, news_score, breakout20, support20)
    trade = build_trade_plan(symbol, close, support20, breakout20, ema20, atr_like, setup)

    return {
        "symbol": symbol,
        "date": date,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "daily_change_percent": day_change,
        "high_52_week": high_52,
        "low_52_week": low_52,
        "ema20": ema20,
        "ema50": ema50,
        "rsi14": rsi14,
        "volume": volume,
        "avg_vol20": avg_vol20,
        "volume_ratio": vol_ratio,
        "breakout_20d": breakout20,
        "support_20d": support20,
        "recent_low": recent_low,
        "recent_high": recent_high,
        "percent_from_recent_low": pct_from_low,
        "percent_from_recent_high": pct_from_high,
        "trend_classification": trend,
        "setup_type": setup,
        "trade_levels": trade,
    }


#Peer
def peer_list(symbol):
    return PEERS.get(symbol.upper(), DEFAULT_PEERS)


def similar_setups(main_result, peers, top_n=5):
    rows = []

    for peer in peers:
        try:
            result = analyze(peer, use_news=False)
            if result is None:
                continue

            score = 0
            if result["setup_type"] == main_result["setup_type"]:
                score += 4

            score += max(0, 3 - abs(result["rsi14"] - main_result["rsi14"]) / 10)
            score += max(0, 2 - abs(result["volume_ratio"] - main_result["volume_ratio"]))
            score += max(0, 2 - abs(result["percent_from_recent_high"] - main_result["percent_from_recent_high"]) / 10)

            rows.append({
                "Ticker": f"{result['symbol']} {peer_icon(result['setup_type'])}",
                "Setup": result["setup_type"],
                "Close": round(result["close"], 2),
                "RSI": round(result["rsi14"], 2) if pd.notna(result["rsi14"]) else None,
                "Volume Ratio": round(result["volume_ratio"], 2),
                "Buy Price": round(result["trade_levels"]["buy_price"], 2),
                "Stop Loss": round(result["trade_levels"]["stop_loss"], 2),
                "Target 1": round(result["trade_levels"]["sell_target_1"], 2),
                "Target 2": round(result["trade_levels"]["sell_target_2"], 2),
                "Similarity Score": round(score, 2),
            })
        except Exception:
            continue

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    return df.sort_values(by=["Similarity Score", "Ticker"], ascending=[False, True]).head(top_n)


#Output
def print_analysis(result):
    symbol_icon = icon_for(result["trend_classification"], result["setup_type"])

    print("\n" + title("Stock Information"))
    print(title("-----------------"))
    print(title(f"Ticker: {result['symbol']} {symbol_icon}"))

    print(info(f"Date: {result['date']}"))
    print(info(f"Open: {round(result['open'], 2)}"))
    print(info(f"High: {round(result['high'], 2)}"))
    print(info(f"Low: {round(result['low'], 2)}"))
    print(info(f"Close: {round(result['close'], 2)}"))

    if result["daily_change_percent"] > 0:
        print(good(f"Daily Change: {signed_pct(result['daily_change_percent'])}"))
    elif result["daily_change_percent"] < 0:
        print(bad(f"Daily Change: {signed_pct(result['daily_change_percent'])}"))
    else:
        print(info(f"Daily Change: {signed_pct(result['daily_change_percent'])}"))

    print(info(f"52-Week High: {round(result['high_52_week'], 2)}"))
    print(info(f"52-Week Low: {round(result['low_52_week'], 2)}"))

    print(good(f"20 EMA: {round(result['ema20'], 2)}") if result["close"] > result["ema20"] else bad(f"20 EMA: {round(result['ema20'], 2)}"))
    print(good(f"50 EMA: {round(result['ema50'], 2)}") if result["ema20"] > result["ema50"] else bad(f"50 EMA: {round(result['ema50'], 2)}"))

    rsi_value = result["rsi14"]
    if pd.notna(rsi_value):
        if 45 <= rsi_value <= 68:
            print(good(f"RSI (14): {round(rsi_value, 2)}"))
        elif rsi_value < 35 or rsi_value > 75:
            print(bad(f"RSI (14): {round(rsi_value, 2)}"))
        else:
            print(info(f"RSI (14): {round(rsi_value, 2)}"))
    else:
        print(info("RSI (14): N/A"))

    print(info(f"Volume: {int(result['volume']) if pd.notna(result['volume']) else 'N/A'}"))
    print(info(f"20-Day Avg Volume: {round(result['avg_vol20'], 0)}") if pd.notna(result["avg_vol20"]) else info("20-Day Avg Volume: N/A"))

    if result["volume_ratio"] >= 1.0:
        print(good(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))
    elif result["volume_ratio"] < 0.8:
        print(bad(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))
    else:
        print(info(f"Volume Ratio: {round(result['volume_ratio'], 2)}"))

    print(info(f"20-Day Breakout Level: {round(result['breakout_20d'], 2)}"))
    print(info(f"20-Day Support Level: {round(result['support_20d'], 2)}"))
    print(info(f"Recent 2-Month Low: {round(result['recent_low'], 2)}"))
    print(info(f"Recent 2-Month High: {round(result['recent_high'], 2)}"))

    if result["percent_from_recent_low"] >= 10:
        print(good(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))
    elif result["percent_from_recent_low"] <= 3:
        print(bad(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))
    else:
        print(info(f"% From Recent Low: {round(result['percent_from_recent_low'], 2)} %"))

    if result["percent_from_recent_high"] >= -5:
        print(good(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))
    elif result["percent_from_recent_high"] <= -15:
        print(bad(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))
    else:
        print(info(f"% From Recent High: {round(result['percent_from_recent_high'], 2)} %"))

    trend = result["trend_classification"]
    if trend == "Bull market-like condition":
        print(good(f"Trend Classification: {trend}"))
    elif trend == "Bear market-like condition":
        print(bad(f"Trend Classification: {trend}"))
    else:
        print(info(f"Trend Classification: {trend}"))

    setup = result["setup_type"]
    if setup in {"Support Bounce", "Breakout Forming", "Bullish Pullback"}:
        print(good(f"Swing Setup: {setup}"))
    elif setup in {"Weak Near Support", "Bearish / Weak Trend"}:
        print(bad(f"Swing Setup: {setup}"))
    else:
        print(info(f"Swing Setup: {setup}"))

    trade = result["trade_levels"]
    print("\n" + title("Swing Trade Levels"))
    print(title("------------------"))
    print(info(f"Buy Price: {round(trade['buy_price'], 2)}"))
    print(bad(f"Stop Loss: {round(trade['stop_loss'], 2)}"))
    print(good(f"Sell Target 1 (Short Term): {round(trade['sell_target_1'], 2)}"))
    print(good(f"Sell Target 2 (Long Term): {round(trade['sell_target_2'], 2)}"))


#Main
symbol = input("Enter stock ticker: ").strip().upper()

try:
    main_result = analyze(symbol, use_news=True)

    if main_result is None:
        print("Stock not found or no data available.")
    else:
        print_analysis(main_result)

        peers = [ticker for ticker in peer_list(symbol) if ticker != symbol]
        peer_df = similar_setups(main_result, peers, top_n=5)

        print("\n" + title("Similar Ticker Setups To Look Up"))
        print(title("--------------------------------"))
        if peer_df.empty:
            print(info("No similar ticker setups found."))
        else:
            print(peer_df.to_string(index=False))

except Exception as exc:
    print("Error fetching stock data or processing analysis.")
    print("Details:", exc)

Enter stock ticker: AAPL

Stock Information
-----------------
Ticker: AAPL 🟢
Date: 2026-03-06
Open: 258.63
High: 258.77
Low: 254.37
Close: 257.46
Daily Change: -0.45%
52-Week High: 288.62
52-Week Low: 168.63
20 EMA: 264.71
50 EMA: 265.09
RSI (14): 51.92
Volume: 41120042
20-Day Avg Volume: 45889026.0
Volume Ratio: 0.9
20-Day Breakout Level: 280.9
20-Day Support Level: 254.37
Recent 2-Month Low: 243.42
Recent 2-Month High: 280.9
% From Recent Low: 5.77 %
% From Recent High: -8.35 %
Trend Classification: Neutral / no clear bull-bear condition
Swing Setup: Support Bounce

Swing Trade Levels
------------------
Buy Price: 257.46
Stop Loss: 251.38
Sell Target 1 (Short Term): 263.54
Sell Target 2 (Long Term): 266.58

Similar Ticker Setups To Look Up
--------------------------------
 Ticker             Setup  Close   RSI  Volume Ratio  Buy Price  Stop Loss  Target 1  Target 2  Similarity Score
  QQQ 🟢    Support Bounce 599.83 48.55          1.21     599.83     587.25    608.33    618.70        